In [1]:
# imports for minimodel
import os
import copy
import numpy as np
import torch
import argparse
from minimodel import data
from minimodel import model_builder
from minimodel import model_trainer
from minimodel import model_trainer_exp
from minimodel import metrics

In [2]:
import torch
import torchvision
print("torch:", torch.__version__, "cuda:", torch.version.cuda)
print("torchvision:", torchvision.__version__)
print("cuda available:", torch.cuda.is_available())


torch: 2.5.1 cuda: 12.4
torchvision: 0.20.1
cuda available: True


In [ ]:
# additional imports for experanto
from pathlib import Path
import matplotlib.pyplot as plt

from tqdm import tqdm
from omegaconf import OmegaConf, open_dict

from experanto.datasets import ChunkDataset
from experanto.dataloaders import get_multisession_dataloader

In [4]:
cfg = {
    "dataset": {
        "global_sampling_rate": None,
        "global_chunk_size": None,
        "add_behavior_as_channels": False,
        "replace_nans_with_means": False,
        "cache_data": False,
        "out_keys": [
            "screen",
            "responses",
            "timestamps",
        ],
        "normalize_timestamps": True,
        "safe_interval_threshold": 0.0,                 # um alle datenpunkte zu bekommen default war 0.5
        "modality_config": {
            "screen": {
                "keep_nans": False,
                "sampling_rate": 7.496251874,
                "chunk_size": 1,                        # ich habe glaub ich keinen zeitlichen zusammenhang zwischen den samples
                "valid_condition": {
                    "tier": "train",
                },
                "offset": 0,
                "sample_stride": 1,
                "include_blanks": False,
                "transforms": {
                    "normalization": "normalize",
                    "Resize": {
                        "_target_": "torchvision.transforms.v2.Resize",
                        "size": [66, 130],
                    },
                },
                "interpolation": {
                    "rescale": True,
                    "rescale_size": [66, 130],
                },
            },
            "responses": {
                "keep_nans": False,
                "sampling_rate": 7.496251874,
                "chunk_size": 1,                
                "offset": 0.0,
                "transforms": {
                    "normalization": "standardize",
                },
                "interpolation": {
                    "interpolation_mode": "nearest_neighbor",
                },
                "filters": {
                    "nan_filter": {
                        "__target__": "experanto.filters.common_filters.nan_filter",
                        "__partial__": True,
                         "vicinity": 0.05,
                    },
                },
            },
        },
    },
    "dataloader": {
        "batch_size": 16,
        "shuffle": True,
        "num_workers": 4,
        "pin_memory": True,
        "drop_last": False,
        "prefetch_factor": 2,
    },
}


In [5]:
cfg_train = copy.deepcopy(cfg)
cfg_val = copy.deepcopy(cfg)
cfg_test = copy.deepcopy(cfg)

cfg_train = OmegaConf.create(cfg_train)
cfg_val = OmegaConf.create(cfg_val)
cfg_test = OmegaConf.create(cfg_test)

In [6]:
cfg_train.dataset.modality_config.screen.valid_condition = {"tier": "train"}
cfg_val.dataset.modality_config.screen.valid_condition = {"tier": "validation"}

cfg_test.dataset.modality_config.screen.valid_condition = {"tier": "test"}
cfg_test.dataset.out_keys.append("image_id")        # I sadly need this to combine all samples with same image_id
cfg_test.dataloader.drop_last = False               # Here I dont need the batches to be the same size
cfg_test.dataloader.shuffle = False

In [7]:
paths = ["/mnt/vast-nhr/projects/bthesis_cidas_richter/benjamin/minimodel/internship/data_experanto/nat30k_L1_A5_022723_experanto"]
train_dl = get_multisession_dataloader(paths, cfg_train)
val_dl = get_multisession_dataloader(paths, cfg_val)
test_dl = get_multisession_dataloader(paths, cfg_test)

/user/benjamin.richter02/u23846/.conda/envs/mini-exp-venv2/lib/python3.10/site-packages/torchvision/transforms/v2/_deprecated.py:42: UserWarning: The transform `ToTensor()` is deprecated and will be removed in a future release. Instead, please use `v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])`.Output is equivalent up to float precision.
  warnings.warn(


In [8]:
dataset_name, batch = next(iter(test_dl))
print(
    f"dataset: {dataset_name}",
)
for image_id, v in batch.items():
    # print(f"modality: {k}, shape: {v.shape}")

    print("Modality: ", image_id)
    if hasattr(v, "shape"):
        print("Shape:", v.shape)
    elif isinstance(v, dict):
        print("Sub-dict keys:", list(v.keys()))
    else:
        print("Type:", type(v))
    print()

# video shape: batch, times, channels, height, width    -> here screen batch * height * channels or times * width ? 
# neuronal responses: batch, times, neurons

dataset: session_0
Modality:  responses
Shape: torch.Size([16, 1, 6636])

Modality:  screen
Shape: torch.Size([16, 66, 1, 130])

Modality:  timestamps
Sub-dict keys: ['responses', 'screen']

Modality:  image_id
Shape: torch.Size([16])



In [9]:
def count_samples(dl):
    n = 0
    for _, batch in dl:
        n += batch["responses"].shape[0]
    return n

In [10]:
if cfg_train.dataloader.drop_last:  train_dl_length = len(train_dl) * cfg_train.dataloader.batch_size
else:                               train_dl_length = count_samples(train_dl)
if cfg_val.dataloader.drop_last:    val_dl_length = len(val_dl) * cfg_val.dataloader.batch_size
else:                               val_dl_length = count_samples(val_dl)
if cfg_test.dataloader.drop_last:   test_dl_length = len(test_dl) * cfg_test.dataloader.batch_size
else:                               test_dl_length = count_samples(test_dl)

batch_size = cfg_val.dataloader.batch_size  # nur im val_epoch benötigt

print("length of train_dl: ", train_dl_length)
print("length of val_dl: ", val_dl_length)
print("length of test_dl: ", test_dl_length)

length of train_dl:  24779
length of val_dl:  2754
length of test_dl:  4906


In [11]:
_ ,batch = next(iter(train_dl))
NN = batch["responses"].shape[-1]       # number of neurons
print("number of neurons: ", NN)

number of neurons:  6636


In [12]:
# setup
device = torch.device('cuda')
mouse_id = 0
weight_path = './checkpoints_16-320_exp'
results_path = './results_16-320_exp'
os.makedirs(weight_path, exist_ok=True)
os.makedirs(results_path, exist_ok=True)


In [13]:
# Building Model

nlayers = 2
nconv1 = 16
nconv2 = 320
model, in_channels = model_builder.build_model(NN=NN, n_layers=nlayers, n_conv=nconv1, n_conv_mid=nconv2)
model_name = model_builder.create_model_name(data.mouse_names[mouse_id], data.exp_date[mouse_id], n_layers=nlayers, in_channels=in_channels)

model_path = os.path.join(weight_path, model_name)
print('model path: ', model_path)
model = model.to(device)

core shape:  torch.Size([1, 320, 33, 65])
input shape of readout:  (320, 33, 65)
model name:  l1a5_022723_2layer_16_320_clamp_norm_depthsep_pool.pt
model path:  ./checkpoints_16-320_exp/l1a5_022723_2layer_16_320_clamp_norm_depthsep_pool.pt


In [14]:
# Training the model
print(device)
if not os.path.exists(model_path):
    best_state_dict = model_trainer_exp.train(model, train_dl=train_dl, val_dl=val_dl, 
                                              train_dl_length=train_dl_length, val_dl_length=val_dl_length, 
                                              n_neurons=NN, batch_size=batch_size ,device=device)
    torch.save(best_state_dict, model_path)
    print('saved model', model_path)
model.load_state_dict(torch.load(model_path))
print('loaded model', model_path)

cuda
Learning rate = 0.001


KeyboardInterrupt: 

In [ ]:
import numpy as np
import torch
from collections import defaultdict

def build_img_test_and_spks_rep_all(test_dl, device="cpu"):
    reps = defaultdict(list)      # image_id -> list of (n_neurons,) numpy arrays
    img_by_id = {}                # image_id -> img , torch tensor (1,66,130)

    for _, batch in test_dl:
        spks_batch = batch["responses"]     # (B,1,N)
        img_batch = batch["screen"]         # (B,66,1,130) or similar
        img_ids = batch["image_id"]          # (B,)

        # ---- to CPU for storage ----
        spks_batch = spks_batch.squeeze().detach().cpu().numpy()    # (B,1,N) -> (B,N)
        
        img_batch = img_batch.squeeze().unsqueeze(1)    # (B,66,1,130) -> (B,1,66,130)
        img_batch = img_batch.detach().cpu()
        
        img_ids = img_ids.detach().cpu().numpy()

        # ---- group by image_id ----
        for i in range(len(img_ids)):
            image_id = int(img_ids[i])
            reps[image_id].append(spks_batch[i])

            # store first occurrence of the image
            if image_id not in img_by_id:
                img_by_id[image_id] = img_batch[i]  # (1,66,130)

    # stable ordering
    unique_ids = sorted(reps.keys())

    # img_test: (n_unique,1,66,130)
    img_test = torch.stack([img_by_id[k] for k in unique_ids], dim=0).to(device)

    # spks_rep_all
    spks_rep_all = np.empty(len(unique_ids), dtype=object)  # np.array (n_unique, )
    for i, image_id in enumerate(unique_ids):
        spks_rep_all[i] = np.stack(reps[image_id], axis=0)  # one sample of np.array (n_repeats, n_neurons)

    return img_test, spks_rep_all, unique_ids


In [ ]:
img_test, spks_rep_all, unique_ids = build_img_test_and_spks_rep_all(test_dl, device=device)

In [ ]:
print("Total test images used: ", len(unique_ids))
print("img_test: ", img_test.shape)
print("spks_rep_all: ", spks_rep_all.shape)


Total test images used:  500
img_test:  torch.Size([500, 1, 66, 130])
spks_rep_all:  (500,)


In [ ]:
# test model
test_pred = model_trainer.test_epoch(model, img_test)
print('test_pred: ', test_pred.shape, test_pred.min(), test_pred.max())


test_fev, test_feve = metrics.feve(spks_rep_all, test_pred)
print('FEVE (test, all): ', np.mean(test_feve))

threshold = 0.15
print(f'filtering neurons with FEV > {threshold}')
valid_idxes = np.where(test_fev > threshold)[0]
print(f'valid neurons: {len(valid_idxes)} / {len(test_fev)}')
print(f'FEVE (test, FEV>0.15): {np.mean(test_feve[test_fev > threshold])}')

test_pred:  (500, 6636) 0.9992936 1.0007973
FEVE (test, all):  -1.4638102706683151
filtering neurons with FEV > 0.15
valid neurons: 4246 / 6636
FEVE (test, FEV>0.15): -0.9367524548931532


In [ ]:
# ---- Saving performance scores ----
file_name = "results_" + str(mouse_id)
results_file_path = os.path.join(results_path, file_name)

print(f"Results saved at: {results_file_path}")
ineur = np.arange(0, NN)
np.savez(results_file_path, FEV_scores=test_fev, FEVE_scores=test_feve, neurons_index=ineur)

Results saved at: ./results_16-320_exp/results_0


NameError: name 'ineur' is not defined